In [47]:
import pandas as pd 
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_absolute_error , mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

In [48]:
loan_data=pd.read_csv("loan_approval_data.csv")

In [49]:
loan_data.head()
# loan_data.info()
# loan_data.describe()
 
# loan_data.isnull().sum()
# loan_data.duplicated().any()


,Applicant_ID,Applicant_Income,Coapplicant_Income,Employment_Status,Age,Marital_Status,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,Loan_Term,Loan_Purpose,Property_Area,Education_Level,Gender,Employer_Category,Loan_Approved
0,1.0,17795.0,1387.0,Salaried,51.0,Married,0.0,637.0,4.0,0.53,19403.0,45638.0,16619.0,84.0,Personal,Urban,Not Graduate,Female,Private,No
1,2.0,2860.0,2679.0,Salaried,46.0,Married,3.0,621.0,2.0,0.30,2580.0,49272.0,38687.0,NaN,Car,Semiurban,Graduate,NaN,Private,No
2,3.0,7390.0,2106.0,Salaried,25.0,Single,2.0,674.0,4.0,0.20,13844.0,6908.0,27943.0,72.0,NaN,Urban,NaN,Female,Government,Yes
3,4.0,13964.0,8173.0,Salaried,40.0,Married,2.0,579.0,3.0,0.31,9553.0,10844.0,27819.0,60.0,Business,Rural,Graduate,Female,Government,No
4,5.0,13284.0,4223.0,Self-employed,31.0,Single,2.0,721.0,1.0,0.29,9386.0,37629.0,12741.0,72.0,Car,NaN,Graduate,Male,Private,Yes


In [50]:
# for missing catagorical data fill with Mode(Max Appeared)
categorical_cols = [
    "Employment_Status",
    "Marital_Status",
    "Loan_Purpose",
    "Property_Area",
    "Education_Level",
    "Gender",
    "Employer_Category",
    "Loan_Approved"
    
]

for col in categorical_cols:
    loan_data[col] = loan_data[col].fillna(loan_data[col].mode()[0])

In [51]:
# for missing numerical data with (Mean(symmetric data) / Median(When there is outlier or skewed data))
numerical_cols = [
    "Applicant_Income",
    "Coapplicant_Income",
    "Age",
    "Dependents",
    "Credit_Score",
    "Existing_Loans",
    "DTI_Ratio",
    "Savings",
    "Collateral_Value",
    "Loan_Amount",
    "Loan_Term"
    
]

for col in numerical_cols:
    loan_data[col] = loan_data[col].fillna(loan_data[col].median())

In [52]:
# Drop Id -> Non -Meaningful
loan_data.drop("Applicant_ID", axis=1, inplace=True)

In [53]:
# 2. Convert binary categorical columns to 0/1
loan_data["Gender"] = loan_data["Gender"].map({
    "Female": 0,
    "Male": 1
})

loan_data["Marital_Status"] = loan_data["Marital_Status"].map({
    "Single": 0,
    "Married": 1
})

loan_data["Education_Level"] = loan_data["Education_Level"].map({
    "Not Graduate": 0,
    "Graduate": 1
})

# Target variable
loan_data["Loan_Approved"] = loan_data["Loan_Approved"].map({
    "No": 0,
    "Yes": 1
})


# 3. One-hot encode nominal categorical columns
nominal_cols = [
    "Employment_Status",
    "Loan_Purpose",
    "Property_Area",
    "Employer_Category"
]

loan_data = pd.get_dummies(
    loan_data,
    columns=nominal_cols,
    drop_first=True
)

In [54]:
x=loan_data.drop('Loan_Approved', axis=1)
y=loan_data['Loan_Approved']

In [55]:

x_train,x_test,y_train,y_test= train_test_split(x, y, test_size=0.2, random_state=42)


In [56]:
# Scaling - Standerization - Using Z-score (update pameter by making mean=0 && Standard Deviation =1) Ex. salray=200000 and age =23 not comparable we have scale it
# z(new parameter) = (x(data point) - mean ) / Standard Deviation 

# Scaling -> Normalization - Using Min-Max-Score (make Mean=0 && S.D.=0 ) 
# x = (x - x_Min) / (x_Max - x_Min)

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

scaler1 = MinMaxScaler()

scaler2= StandardScaler()

x_train_scaled= scaler2.fit_transform(x_train)
x_test_scaled=  scaler2.transform(x_test)

In [62]:
#  Linear Regression Model

model= LinearRegression()

model.fit(x_train_scaled,y_train)
y_predict= model.predict(x_test_scaled)

y_pred_class = (y_predict >= 0.5).astype(int)

print("Linear Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_class) * 100)
print("Precision:", precision_score(y_test, y_pred_class) * 100)
print("Recall:", recall_score(y_test, y_pred_class) * 100)
print("F1 Score:", f1_score(y_test, y_pred_class) * 100)

# L1 regularization 

from sklearn.linear_model import Lasso

model = Lasso(max_iter=100000)

param_grid = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="r2"
)

grid.fit(x_train_scaled, y_train)

print("Best Alpha:", grid.best_params_["alpha"])
print("Best CV R2 Score:", grid.best_score_)

y_predict = grid.predict(x_test_scaled)

y_pred_class = (y_predict >= 0.5).astype(int)

print("\nLinear Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_class) * 100)
print("Precision:", precision_score(y_test, y_pred_class) * 100)
print("Recall:", recall_score(y_test, y_pred_class) * 100)
print("F1 Score:", f1_score(y_test, y_pred_class) * 100)

#  l2 Regularization 

from sklearn.linear_model import Ridge
model = Ridge()

param_grid = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="r2"
)

grid.fit(x_train_scaled, y_train)

print("Best Alpha:", grid.best_params_["alpha"])
print("Best CV R2 Score:", grid.best_score_)

y_predict = grid.predict(x_test_scaled)

y_pred_class = (y_predict >= 0.5).astype(int)

print("\n Linear Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_class) * 100)
print("Precision:", precision_score(y_test, y_pred_class) * 100)
print("Recall:", recall_score(y_test, y_pred_class) * 100)
print("F1 Score:", f1_score(y_test, y_pred_class) * 100)


Linear Regression
Accuracy: 85.0
Precision: 75.40983606557377
Recall: 75.40983606557377
F1 Score: 75.40983606557377
Best Alpha: 0.01
Best CV R2 Score: 0.40751001885663485

Linear Regression
Accuracy: 85.5
Precision: 78.57142857142857
Recall: 72.1311475409836
F1 Score: 75.21367521367522
Best Alpha: 10
Best CV R2 Score: 0.4001040434072595

 Linear Regression
Accuracy: 85.0
Precision: 75.40983606557377
Recall: 75.40983606557377
F1 Score: 75.40983606557377


In [58]:
# Logistic Regrssion Model (Default with L2 regularization )

model= LogisticRegression(max_iter=1000000)

model.fit(x_train_scaled,y_train)
y_predict= model.predict(x_test_scaled)

print("Accuracy-Score:", accuracy_score(y_test,y_predict)*100, "%")
print("Precision-Score:", precision_score(y_test,y_predict)*100, "%")
print("Recall-Score:", recall_score(y_test,y_predict)*100, "%")
print("F1-Score-Score:", f1_score(y_test,y_predict)*100, "%")
print("Confusion-Matrix:", confusion_matrix(y_test,y_predict))
print("Classification-Report:", classification_report(y_test,y_predict))


# Logistic Regression with L1 Regularization

model_l1 = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    max_iter=1000000
)

model_l1.fit(x_train_scaled, y_train)

y_predict_l1 = model_l1.predict(x_test_scaled)

print("L1 Logistic Regression")
print("Accuracy-Score:", accuracy_score(y_test, y_predict_l1) * 100, "%")
print("Precision-Score:", precision_score(y_test, y_predict_l1) * 100, "%")
print("Recall-Score:", recall_score(y_test, y_predict_l1) * 100, "%")
print("F1-Score:", f1_score(y_test, y_predict_l1) * 100, "%")
print("Confusion-Matrix:\n", confusion_matrix(y_test, y_predict_l1))
print("Classification-Report:\n", classification_report(y_test, y_predict_l1))

Accuracy-Score: 86.5 %
Precision-Score: 78.33333333333333 %
Recall-Score: 77.04918032786885 %
F1-Score-Score: 77.68595041322314 %
Confusion-Matrix: [[126  13]
 [ 14  47]]
Classification-Report:               precision    recall  f1-score   support

           0       0.90      0.91      0.90       139
           1       0.78      0.77      0.78        61

    accuracy                           0.86       200
   macro avg       0.84      0.84      0.84       200
weighted avg       0.86      0.86      0.86       200

L1 Logistic Regression
Accuracy-Score: 87.0 %
Precision-Score: 78.68852459016394 %
Recall-Score: 78.68852459016394 %
F1-Score: 78.68852459016394 %
Confusion-Matrix:
 [[126  13]
 [ 13  48]]
Classification-Report:
               precision    recall  f1-score   support

           0       0.91      0.91      0.91       139
           1       0.79      0.79      0.79        61

    accuracy                           0.87       200
   macro avg       0.85      0.85      0.85     

In [59]:
# Naive-Bayes Model

model= GaussianNB()

model.fit(x_train_scaled,y_train)
y_predict= model.predict(x_test_scaled)

print("Accuracy-Score:", accuracy_score(y_test,y_predict)*100, "%")
print("Precision-Score:", precision_score(y_test,y_predict)*100, "%")
print("Recall-Score:", recall_score(y_test,y_predict)*100, "%")
print("F1-Score-Score:", f1_score(y_test,y_predict)*100, "%")
print("Confusion-Matrix:", confusion_matrix(y_test,y_predict))
print("Classification-Report:", classification_report(y_test,y_predict))


Accuracy-Score: 86.5 %
Precision-Score: 80.35714285714286 %
Recall-Score: 73.77049180327869 %
F1-Score-Score: 76.92307692307693 %
Confusion-Matrix: [[128  11]
 [ 16  45]]
Classification-Report:               precision    recall  f1-score   support

           0       0.89      0.92      0.90       139
           1       0.80      0.74      0.77        61

    accuracy                           0.86       200
   macro avg       0.85      0.83      0.84       200
weighted avg       0.86      0.86      0.86       200



In [60]:
# KNN -Model
 

# Choose Best k -----> Manually

from sklearn.model_selection import cross_val_score

k_grid = [3, 5, 7, 9]
ans = {}

for k in k_grid:

    model = KNeighborsClassifier(n_neighbors=k)

    scores = cross_val_score(
        model,
        x_train_scaled,
        y_train,
        cv=5,
        scoring="recall"
    )

    ans[k] = scores.mean()

best_k = max(ans, key=ans.get)

final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(x_train_scaled, y_train)

y_pred = final_model.predict(x_test_scaled)







    
# GirdSearch ---> used for choose best k automatic 

from  sklearn.model_selection import GridSearchCV

classifier_model=KNeighborsClassifier()

param_grid= {"n_neighbors": [3,5,7,9]}

classifierCV = GridSearchCV(  #classifierCV ->gridsearch object
        classifier_model, # estimetor or model
        param_grid,       # hyperparameter && candidate val
        cv=5   , # folds
        scoring ="recall"  # default accuracy
    )

classifierCV.fit(x_train_scaled,y_train)
y_predict = classifierCV.predict(x_test_scaled)

# results 
print(classifierCV.best_params_)  # return best k
print((classifierCV.best_score_)*100)  # return best score

res= pd.DataFrame(classifierCV.cv_results_)
print(res[["param_n_neighbors","mean_test_score"]])

print("Accuracy-Score:", accuracy_score(y_test,y_predict)*100, "%")
print("Precision-Score:", precision_score(y_test,y_predict)*100, "%")
print("Recall-Score:", recall_score(y_test,y_predict)*100, "%")
print("F1-Score-Score:", f1_score(y_test,y_predict)*100, "%")
print("Confusion-Matrix:", confusion_matrix(y_test,y_predict))
print("Classification-Report:", classification_report(y_test,y_predict))

# "n_neighbors" is the parameter name defined by KNeighborsClassifier.

# This tells GridSearchCV:"Now perform the hyperparameter search using my training data."

# It will:
# 1. Take your param_grid:
#    {"n_neighbors": [3, 5, 7, 9]}
# 2. Try K = 3, 5, 7, 9.
# 3. For each K, perform your cv=5 cross-validation.
# 4. Calculate the average validation score for each K.
# 5. Select the best K.
# For example:
# K = 3 → 86%
# K = 5 → 90%  ← best
# K = 7 → 88%
# K = 9 → 85%

# Then classifierCV stores the results and the best model.

# classifierCV is still a GridSearchCV object. Inside it, GridSearchCV stores several things:
# classifierCV
# │
# ├── best_params_        → best K
# ├── best_score_         → best CV score
# ├── best_estimator_     → model with best K
# └── cv_results_         → results of ALL K values


{'n_neighbors': 3}
55.65602836879433
   param_n_neighbors  mean_test_score
0                  3         0.556560
1                  5         0.489716
2                  7         0.484840
3                  9         0.446631
Accuracy-Score: 74.5 %
Precision-Score: 58.06451612903226 %
Recall-Score: 59.01639344262295 %
F1-Score-Score: 58.536585365853654 %
Confusion-Matrix: [[113  26]
 [ 25  36]]
Classification-Report:               precision    recall  f1-score   support

           0       0.82      0.81      0.82       139
           1       0.58      0.59      0.59        61

    accuracy                           0.74       200
   macro avg       0.70      0.70      0.70       200
weighted avg       0.75      0.74      0.75       200

